[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/33_beam_search.ipynb)

# 🟠 中等: 束搜索解码

实现**束搜索**——经典的序列生成解码算法。

### 函数签名
```python
def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token) -> list[int]:
    # log_prob_fn: 接收 token 列表，返回 (V,) log 概率
    # 返回: 最佳序列 (整数列表)
```

### 算法
1. 从 `[(0.0, [start_token])]` 开始
2. 每一步：用 top-k 个下一个 token 扩展每个束
3. 按总 log 概率保留前 `beam_width` 个束
4. 当最佳束以 `eos_token` 结束或达到 `max_len` 时停止

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✏️ 在此实现你的代码

def beam_search(log_prob_fn, start_token, max_len, beam_width, eos_token):
    pass  # 维护束, 扩展, 剪枝, 返回最佳结果

### 为什么要使用对数概率?

当我们计算多个token的概率乘积时:\
P(sequence) = P(token1) × P(token2) × ... × P(tokenn)

使用对数后:\
log P(sequence) = log P(token1) + log P(token2) + ... + log P(tokenn)

在机器学习中，通常使用自然对数,只要所有计算使用相同的底数，log 概率都是负数, 但比较结果就一致。

In [ ]:
import numpy as np
from typing import List, Tuple, Callable

def beam_search(log_prob_fn: Callable[[List[int]], np.ndarray], 
                start_token: int, 
                max_len: int, 
                beam_width: int, 
                eos_token: int) -> List[int]:
    """
    束搜索解码算法
    原理: 每次解码添加 beam_width 个候选序列作为下一个 token, 并且叠加分数, 知道搜索到 beam_width 个序列或者 max_len 个 token 后停止,最后选取分数最大的序列作为结果
    弊端: 长度偏差，由于分数是根据对数概率叠加计算的,越长的 token, 分数越低。如果是以正值的概率叠加计算，越长的 topent, 分数会变高。
    
    参数:
        log_prob_fn: 接收 token 列表，返回 log 概率的函数, (V,)
        start_token: 起始 token, 一般是 <|im_start|>, ids 为 0
        max_len: 最大序列长度
        beam_width: 束宽度, 表示每次搜索 top_k 个候选 token, 最后输出 beam_width 个序列
        eos_token: 结束 token
    
    返回:
        最佳序列 (整数列表)
    """
    # 初始化：从起始 token 开始，log 概率为 0
    beams = [(0.0, [start_token])]  # (累计对数概率, token 序列)
    
    completed_beams = [] # 已完成的序列(<eos> 结束的序列)
    
    for step in range(max_len - 1):  # -1 因为已经有一个起始token, 最大生成长度
        if len(completed_beams) >= beam_width:
            break
            
        all_candidates = [] # 刷新候选列表
        
        for score, seq in beams:
            ''' 遍历每个束，得到新的候选序列列表 '''
            # 如果当前序列已经以 <eos> 结束，直接添加到完成列表
            if seq[-1] == eos_token:
                completed_beams.append((score, seq))
                continue
            
            log_probs = log_prob_fn(seq)  # 输入序列，获得下一个词的对数概率分布 (V,)
            
            top_k_indices = np.argsort(log_probs)[-beam_width:][::-1] # 获取 top-k 个下一个 token 的索引, [::-1] 进行降序
            
            # 遍历每个候选序列，添加进候选列表中
            for token_idx in top_k_indices:
                token_prob = log_probs[token_idx]
                new_score = score + token_prob # 计算新的得分 sum(log_probs)
                new_seq = seq + [int(token_idx)]
                all_candidates.append((new_score, new_seq))
        
        # 如果没有候选，即每个束最后都是 EOS，提前停止
        if not all_candidates:
            break
        
        # 按分数排序，beams 保留前 beam_width 个
        all_candidates.sort(key=lambda x: x[0], reverse=True)
        beams = all_candidates[:beam_width]
        
        # 检查是否有完成的序列添加进完成列表中
        new_completed = [(score, seq) for score, seq in beams if seq[-1] == eos_token]
        completed_beams.extend(new_completed)
        
        # 如果所有 beam 都完成了，提前停止
        if len(completed_beams) >= beam_width:
            break
        
        # 只保留未完成的束，继续获得下一词的概率分布
        beams = [(score, seq) for score, seq in beams if seq[-1] != eos_token]
        
        # 如果所有束都已完成但没有足够多的完成序列，提前结束
        if not beams and step < max_len - 1:
            break
    
    # 选择最佳序列
    all_beams = completed_beams + beams
    if all_beams:
        # 按分数排序
        all_beams.sort(key=lambda x: x[0], reverse=True)
        return all_beams[0][1]
    else:
        return [start_token]


In [ ]:
# 🧪 调试
def simple_fn(tokens):
    lp = torch.full((5,), -10.0)
    lp[min(len(tokens), 4)] = 0.0
    return lp
seq = beam_search(simple_fn, start_token=0, max_len=5, beam_width=2, eos_token=4)
print('序列:', seq)

In [ ]:
# ✅ 提交
from torch_judge import check
check('beam_search')